# Notebook 1 – Étude du bruit dans les images numériques

**PFE 2025/2026** – Segmentation d'images en présence de bruit  
**Encadrant :** Leila Essannouni

---

Ce notebook couvre :
1. Génération et visualisation du bruit Gaussien
2. Génération et visualisation du bruit Sel-et-Poivre
3. Mesures quantitatives : PSNR et SNR
4. Effet du niveau de bruit sur la qualité visuelle

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from noise import add_gaussian_noise, add_salt_and_pepper_noise, compute_psnr, compute_snr

print('Modules chargés avec succès ✓')

## 1. Image de test
Nous utilisons l'image de référence classique **Lena** (ou une image de votre dataset si disponible).

In [ ]:
import urllib.request
from pathlib import Path

# Génère une image synthétique si aucune image n'est disponible
def make_test_image(size=256):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    # Fond gris
    img[:] = 50
    # Cercle blanc
    cv2.circle(img, (size//2, size//2), size//3, (200,200,200), -1)
    # Rectangle sombre
    cv2.rectangle(img, (20, 20), (80, 80), (30, 30, 30), -1)
    # Texte simulé
    cv2.putText(img, 'PFE', (size//2-30, size-30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
    return img

# Chercher une image dans data/images/ sinon image synthétique
data_dir = Path('../data/images')
imgs = list(data_dir.glob('*.png')) + list(data_dir.glob('*.jpg'))
if imgs:
    image = cv2.imread(str(imgs[0]))
    print(f'Image chargée : {imgs[0].name} | shape={image.shape}')
else:
    image = make_test_image()
    print('Image synthétique générée.')

plt.figure(figsize=(4, 4))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('Image originale')
plt.axis('off')
plt.show()

## 2. Bruit Gaussien
Le bruit Gaussien (additif) suit une distribution normale $\mathcal{N}(0, \sigma^2)$.  
Plus $\sigma$ est grand, plus la dégradation est importante.

In [ ]:
sigmas = [0, 10, 25, 50, 75, 100]

fig, axes = plt.subplots(1, len(sigmas), figsize=(16, 3))
for ax, sigma in zip(axes, sigmas):
    if sigma == 0:
        noisy = image.copy()
        title = 'Original'
    else:
        noisy = add_gaussian_noise(image, sigma=sigma)
        psnr  = compute_psnr(image, noisy)
        title = f'σ={sigma}\nPSNR={psnr:.1f} dB'
    ax.imshow(cv2.cvtColor(noisy, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=8)
    ax.axis('off')

fig.suptitle('Impact du bruit Gaussien', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/bruit_gaussien.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Bruit Sel-et-Poivre
Le bruit Sel-et-Poivre corrompt aléatoirement certains pixels avec des valeurs extrêmes (0 = poivre, 255 = sel).  
La densité $d$ représente la proportion de pixels corrompus.

In [ ]:
densities = [0, 0.02, 0.05, 0.10, 0.20, 0.40]

fig, axes = plt.subplots(1, len(densities), figsize=(16, 3))
for ax, d in zip(axes, densities):
    if d == 0:
        noisy = image.copy()
        title = 'Original'
    else:
        noisy = add_salt_and_pepper_noise(image, density=d)
        psnr  = compute_psnr(image, noisy)
        title = f'd={d}\nPSNR={psnr:.1f} dB'
    ax.imshow(cv2.cvtColor(noisy, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=8)
    ax.axis('off')

fig.suptitle('Impact du bruit Sel-et-Poivre', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/bruit_sel_poivre.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Analyse quantitative : PSNR vs niveau de bruit

In [ ]:
import pandas as pd

rows = []
for sigma in [5, 10, 15, 25, 35, 50, 75, 100]:
    noisy = add_gaussian_noise(image, sigma=sigma)
    rows.append({'Type': 'Gaussien', 'Paramètre': sigma, 'PSNR (dB)': compute_psnr(image, noisy),
                 'SNR (dB)': compute_snr(image, noisy)})

for d in [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
    noisy = add_salt_and_pepper_noise(image, density=d)
    rows.append({'Type': 'Sel-et-Poivre', 'Paramètre': d, 'PSNR (dB)': compute_psnr(image, noisy),
                 'SNR (dB)': compute_snr(image, noisy)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

g = df[df['Type']=='Gaussien']
s = df[df['Type']=='Sel-et-Poivre']

ax1.plot(g['Paramètre'], g['PSNR (dB)'], 'b-o', label='Gaussien (σ)')
ax1.set_xlabel('Écart-type σ')
ax1.set_ylabel('PSNR (dB)')
ax1.set_title('PSNR vs σ (Gaussien)')
ax1.grid(True, alpha=0.3)

ax2.plot(s['Paramètre'], s['PSNR (dB)'], 'r-o', label='Sel-et-Poivre (d)')
ax2.set_xlabel('Densité d')
ax2.set_ylabel('PSNR (dB)')
ax2.set_title('PSNR vs densité (Sel-et-Poivre)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/psnr_vs_bruit.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Distribution des pixels avant/après bruit

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
noisy_g = add_gaussian_noise(gray, sigma=50)
noisy_s = add_salt_and_pepper_noise(gray, density=0.10)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(axes,
                           [gray, noisy_g, noisy_s],
                           ['Original', 'Bruit Gaussien σ=50', 'Bruit S&P d=0.10']):
    ax.hist(img.ravel(), bins=256, range=(0,255), color='steelblue', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('Valeur pixel')
    ax.set_ylabel('Fréquence')
    ax.set_xlim(0, 255)

plt.suptitle('Histogrammes des niveaux de gris', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/histogrammes.png', dpi=150, bbox_inches='tight')
plt.show()